In [2]:
import requests
import json
import os
import time

### Parliamentary Session Metadata
Loop through all the parliamentary sessions available on LegisInfo (from 35th to 45th).
Download the JSON and export to the /data/canada_parl directory.

In [ ]:
parl_sess_list = [
    # "35-1",
    # "35-2",
    # "36-1",
    # "36-2",
    "37-1",
    "37-2",
    "37-3",
    "38-1",
    "39-1",
    "39-2",
    "40-1",
    "40-2",
    "40-3",
    "41-1",
    "41-2",
    "42-1",
    "43-1",
    "43-2",
    "44-1",
    "45-1"
]


In [8]:
output_dir = "../data/canada_parl"
os.makedirs(output_dir, exist_ok=True)

for parl_session in parl_sess_list:
    response = requests.get("https://www.parl.ca/legisinfo/en/bills/json?parlsession="+parl_session)
    data = response.json()
    with open(os.path.join(output_dir, "can-"+parl_session+".json"), "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)


### Canada Bill Text


In [ ]:
def fetch_bill_text(parliament, session, bill_type,stage ,bill_number, fmt="xml"):             
    """Fetch bill text from Parliament of Canada DocumentViewer."""
    url = f"https://www.parl.ca/Content/Bills/{parliament}{session}/{bill_type}/{bill_number}/{bill_number}_{stage}/{bill_number}-e.xml"                                                
    print(url)                                                   
    resp = requests.get(url)
    resp.raise_for_status()                                                   
    return resp.text


In [ ]:
# Example: fetch text for all bills in a session                              
session_data = json.load(open("../data/canada_parl/can-39-1.json"))
os.makedirs("../data/canada_bill_text/39-1", exist_ok=True)                   
stage_map = {
    "First reading":  "1",
    "Second reading": "2",
    "Third reading":  "3",
    "Royal assent":   "4",
}

for session in parl_sess_list:
    for bill in session_data:
        if bill["LatestBillTextTypeId"] == 0:                                     
            continue  # no text available                                         
        bill_num = bill["BillNumberFormatted"]
        parl = bill["ParliamentNumber"]                                           
        sess = bill["SessionNumber"]
        stage = stage_map.get(bill["LatestCompletedMajorStageEn"], "1")
        bill_type = "Government" if (bill["BillTypeEn"] == "House Government Bill" or bill["BillTypeEn"] == "Senate Government Bill") else "Private"                             
        out_path = f"../data/canada_bill_text/{parl}-{sess}/{bill_num}.xml"       
        if os.path.exists(out_path):                                              
            continue
        try:                                                                      
            text = fetch_bill_text(parl, sess, bill_type, stage , bill_num, fmt="xml")               
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(text)                                                     
            time.sleep(0.5)  # be polite to the API                             
        except requests.HTTPError as e:                                           
            exit()

https://www.parl.ca/Content/Bills/391/Government/S-2/S-2_4/S-2-e.xml
https://www.parl.ca/Content/Bills/391/Government/S-3/S-3_4/S-3-e.xml
https://www.parl.ca/Content/Bills/391/Government/S-4/S-4_2/S-4-e.xml
https://www.parl.ca/Content/Bills/391/Government/S-5/S-5_4/S-5-e.xml
https://www.parl.ca/Content/Bills/391/Government/S-6/S-6_4/S-6-e.xml
https://www.parl.ca/Content/Bills/391/Private/S-201/S-201_1/S-201-e.xml
https://www.parl.ca/Content/Bills/391/Private/S-202/S-202_2/S-202-e.xml
https://www.parl.ca/Content/Bills/391/Private/S-203/S-203_1/S-203-e.xml
https://www.parl.ca/Content/Bills/391/Private/S-204/S-204_2/S-204-e.xml
https://www.parl.ca/Content/Bills/391/Private/S-205/S-205_3/S-205-e.xml
https://www.parl.ca/Content/Bills/391/Private/S-206/S-206_2/S-206-e.xml
https://www.parl.ca/Content/Bills/391/Private/S-207/S-207_2/S-207-e.xml
https://www.parl.ca/Content/Bills/391/Private/S-208/S-208_1/S-208-e.xml
https://www.parl.ca/Content/Bills/391/Private/S-209/S-209_3/S-209-e.xml
https:/

: 